# DS RAG Embedder v1 — Train & Evaluate on Kaggle

**Domain-specific embeddings for Data Science & ML documentation retrieval**

- Author: Digvijay Waghela
- Model: [waghelad/ds-rag-embedder-v1](https://huggingface.co/waghelad/ds-rag-embedder-v1)
- Dataset: [waghelad/ds-rag-eval-v1](https://huggingface.co/datasets/waghelad/ds-rag-eval-v1)

Enable **GPU** in Settings for faster training.

In [ ]:
!pip install -q sentence-transformers datasets huggingface_hub scikit-learn pandas

In [ ]:
# Clone or upload project; adjust path if needed
# !git clone https://github.com/dgvj-work/ds-rag-embedder-v1.git
# %cd ds-rag-embedder-v1

import sys
from pathlib import Path
ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from scripts.build_corpus import build
stats = build(corpus_size=600)
stats

In [ ]:
from ds_rag_embedder.train import train
from ds_rag_embedder.config import EmbedderConfig
from pathlib import Path

cfg = EmbedderConfig(epochs=4, batch_size=32, output_dir=Path('models/ds-rag-embedder-v1'))
model_path = train(config=cfg)
model_path

In [ ]:
from ds_rag_embedder.evaluate import compare_models
results = compare_models({
    'all-MiniLM-L6-v2': 'sentence-transformers/all-MiniLM-L6-v2',
    'bge-small-en-v1.5': 'BAAI/bge-small-en-v1.5',
    'ds-rag-embedder-v1': 'models/ds-rag-embedder-v1',
})
results

In [ ]:
from ds_rag_embedder import DSRAGEmbedder
embedder = DSRAGEmbedder('models/ds-rag-embedder-v1')
docs = [
    'Use nested cross-validation when tuning hyperparameters.',
    'SMOTE on full data before split causes leakage.',
    'PSI above 0.25 indicates significant drift.',
]
for hit in embedder.search('hyperparameter tuning without overfitting', docs, top_k=3):
    print(hit['score'], hit['document'])

In [ ]:
# Optional: push to Hugging Face (set HF token in Kaggle secrets)
# from huggingface_hub import login
# login(token=UserSecrets.get('HF_TOKEN'))
# embedder.push_to_hub('waghelad/ds-rag-embedder-v1')